# 31. Do any two columns interact?

**A diagnostic, not an experiment.** No training, no `experiments.csv` row, same
treatment as `04`, `05` and `07`. It exists to price an idea before a run is spent on
it, which is the only thing that has ever worked in this repo.

## The sentence this is testing

This repo explains why target encoding was the one feature idea that paid, and the
explanation contains a clause that was never measured:

> Why it works here when no other feature idea did: with 12 columns and **no
> interactions to discover**, the model's problem was never which quantities to look at.

That clause is load bearing. It is the reason feature engineering is listed as rejected,
and the reason the board was declared closed twice. It is also an assertion. Nobody has
ever target encoded a **pair** of columns here, so "there are no interactions" has the
same status that "CatBoost loses" had on 2026-08-16 and that "XGBoost will behave like
CatBoost" had on 2026-08-18: a conclusion with nothing under it.

Three rejections in this competition have already been overturned. All three rested on a
premise that had quietly stopped being true, or had never been checked. This notebook
checks this one.

## Why the prior is genuinely uncertain, in both directions

**For.** The rule with six points on it says a change of representation revalues
learners and their knobs do not. A crossed encoding is a change of representation, and
it is the same change that paid the first time: this repo argues 1-way target encoding
worked because a level "arrives as one number the tree can split on in a single cut,
where before they cost many". A 2D cell costs a depth-6 tree considerably more than one
cut, so the argument transfers directly and predicts a gain.

**Against.** A gradient boosted tree already models interactions. That is most of what
depth is for, and row 38 runs at depth 6 on 36 features, so every 2-way interaction is
already reachable. If the trees are already finding these cells, handing them the same
information pre-computed buys nothing. This is exactly what happened to the decimal
lattice in row 19: a large, verified, real effect in the data that was worth -0.000132
in the model, because the model already had it.

So the honest prior is that this is the strongest untried idea in the competition and
also the one most likely to reproduce row 19. That is why it gets a diagnostic and not a
five-fold run.

## What is measured, and the trap being avoided

For each of the 66 pairs, on fold 0 only:

1. `auc_cross`, the AUC of the crossed target encoding as a single feature.
2. `auc_add`, the AUC of the best additive combination of the two 1-way encodings,
   computed in logit space.
3. **`lift = auc_cross - auc_add`**, which is the number that matters. It asks what the
   cross knows that the two columns separately do not.

A pair with a large `auc_cross` and no lift is two informative columns, not an
interaction. That distinction is the whole point, and reporting `auc_cross` alone would
manufacture 66 exciting results out of the fact that screen time predicts addiction.

**The negative control, and getting it right took two attempts.** More cells means more
capacity, and capacity can look like signal, so the same measurement is run on pairs
with no interaction in them. The question is how to remove the interaction without
removing anything else.

A plain shuffle of column B is wrong and I ran it before noticing. It destroys B's own
relationship with the target along with the interaction, so the crossed encoding is
wrecked while the additive baseline still has an intact B to lean on. That does not
measure the null, it measures the damage, and it produced a control band running down to
-0.21 that would have accepted almost anything as a real result.

The control used here shuffles B **within each target class**. The joint distribution of
B and the target is preserved exactly, so B's 1-way encoding is statistically identical
to the real one and the additive baseline is just as strong as it was. What is destroyed
is the A-B pairing, which is precisely and only the thing under test.

This is the control `29` taught me to build, on the other side of the question: there
the worry was a knob doing nothing, here it is a statistic doing something on its own.

Scoring is on held-out rows throughout, so the encoder is fit on the training portion
and never sees fold 0.

## The decision rule, written before the run

| lift, best real pair, against the shuffled band | verdict |
|---|---|
| clearly outside it, and above +0.0005 | build the crossed feature set, `32` |
| outside it but small | rank the pairs, take the top handful only |
| inside it | **the clause is verified, feature engineering stays rejected** |

+0.0005 is chosen against a scale rather than picked: 1-way target encoding was worth
+0.003312 and is the largest gain in the competition, so a sixth of it from a single
pair would be a large result. A null here is a genuinely useful outcome, because it
closes the last open question about this feature set with a measurement instead of a
sentence.


In [ ]:
# A diagnostic. One fold, no models, no ledger row.
#
# SMOKE defaults to False here, unlike every other notebook in this repo, and the
# reason is that this one trains nothing. It is groupbys and 66 two-parameter logistic
# fits, so the full 691,369 rows cost a couple of minutes. At 20,000 rows the control
# band is wider than every effect being measured and the run answers nothing, which
# makes a smoke pass actively misleading rather than cheap insurance.
SMOKE = False

SEED = 42
PROBE_FOLD = 0

# Numerics are binned before crossing. 12 quantile bins against 691,369 rows leaves
# roughly 4,800 rows per 2D cell on the widest pairs, which is ample support at
# SMOOTH = 10. `age` has 18 distinct values and is used as-is.
N_BINS = 12
SMOOTH = 10.0
RAW_AS_LEVELS = ("age",)

# The negative control. Enough shuffled pairs to see the band's edge rather than just
# its centre, since the decision compares against the MAXIMUM of the band.
N_SHUFFLE = 30
SHUFFLE_SEED = 12345

EXPECTED_FOLD_SHA = "ec282b0968059676"
LIFT_GATE = 0.0005

print(f"SMOKE = {SMOKE}   {N_BINS} bins   smooth {SMOOTH}   "
      f"{N_SHUFFLE} shuffled control pairs")


## Stage 1. Data, folds, leak checklist

The fold checksum is the only thing standing between an out-of-fold vector that
blends and one that is silently misaligned, so it is checked before anything trains
rather than after.

In [ ]:
import itertools
import ast
import gc
import hashlib
import time
from pathlib import Path

import lightgbm as lgb
import numpy as np
import pandas as pd
from sklearn.metrics import roc_auc_score
from sklearn.model_selection import KFold, StratifiedKFold

# Runs here or on Kaggle. Both are found by name rather than by assuming a shape.
KAG = Path("/kaggle/input")
ON_KAGGLE = KAG.exists()
LOCAL = next((b for b in [Path.cwd(), *Path.cwd().parents]
              if (b / "data" / "raw" / "train.csv").exists()), None)


def locate(name):
    if ON_KAGGLE:
        hits = sorted(KAG.rglob(name))
        if hits:
            return hits[0]
    if LOCAL is not None:
        for d in ("data/raw", "artifacts/oof", "notebooks", "submissions"):
            p = LOCAL / d / name
            if p.exists():
                return p
    raise FileNotFoundError(name)


OUT = Path("/kaggle/working") if ON_KAGGLE else LOCAL / "artifacts" / "oof"
SUB = Path("/kaggle/working") if ON_KAGGLE else LOCAL / "submissions"
print(f"running {'on Kaggle' if ON_KAGGLE else 'locally'}, writing to {OUT}")

train_full = pd.read_csv(locate("train.csv"))
test = pd.read_csv(locate("test.csv"))
TARGET = "addicted_label"
CAT = ["gender", "stress_level", "academic_work_impact"]
COLS = [c for c in train_full.columns if c not in ("id", TARGET)]

# Leak checklist, re-run rather than ticked by inspection. `id` is a contiguous row
# index that separates train from test perfectly, so it is a guaranteed leak if it
# ever reaches the model.
checks = {
    "id is not a feature": "id" not in COLS,
    "target is not a feature": TARGET not in COLS,
    "train and test ids do not overlap":
        not (set(train_full["id"]) & set(test["id"])),
    "train and test feature lists match":
        COLS == [c for c in test.columns if c != "id"],
}
for name, ok in checks.items():
    print(f"  [{'ok' if ok else 'FAIL'}] {name}")
LEAK_OK = all(checks.values())

# ROW_IDX maps this run's rows back into the saved member vectors.
if SMOKE:
    ROW_IDX = np.sort(train_full.sample(20000, random_state=0).index.to_numpy())
    train = train_full.loc[ROW_IDX].reset_index(drop=True)
    test = test.head(5000).reset_index(drop=True)
    N_EST, BENCH_EST = 200, 50
    # Measured 2026-08-19 and written up: at 16,000 rows this machine
    # runs 101x slower at n_jobs=-1 than at n_jobs=1, monotone in the thread count.
    # N_JOBS above is chosen to match row 17 on Kaggle at 691,369 rows, where it is
    # right. A smoke run produces no ledger number, so overriding it here costs
    # nothing and is the difference between two minutes and giving up on the check.
    N_JOBS = 1
else:
    ROW_IDX = np.arange(len(train_full))
    train = train_full

y = train[TARGET].to_numpy()
folds = np.full(len(train), -1, dtype=np.int64)
for i, (_, va) in enumerate(StratifiedKFold(5, shuffle=True,
                                            random_state=SEED).split(train, y)):
    folds[va] = i

sha = hashlib.sha256(folds.tobytes()).hexdigest()[:16]
ALIGNED = sha == EXPECTED_FOLD_SHA
print()
print(f"rows {len(train):,}   target rate {y.mean():.6f}")
print(f"fold sizes {np.bincount(folds).tolist()}")
print(f"fold sha {sha}  expected {EXPECTED_FOLD_SHA}")
if SMOKE:
    print("SMOKE: subsampled, so the sha is EXPECTED to differ. Not a check.")
else:
    print("fold alignment: VERIFIED" if ALIGNED else
          "fold alignment: MISMATCH - the OOF from this run is not blendable")

X = train[COLS].copy()
X_test = test[COLS].copy()
for c in CAT:
    X[c] = X[c].astype("category")
    X_test[c] = X_test[c].astype("category")

In [ ]:
# This diagnostic scores on fold 0 and fits on the rest, the same shape as the CatBoost
# probe in 06. No inner nesting: nothing here is read out on training rows.
tr0 = np.where(folds != PROBE_FOLD)[0]
va0 = np.where(folds == PROBE_FOLD)[0]
print(f"fit on {len(tr0):,} rows, score on {len(va0):,}")


## Stage 2. Levels, encoders, and the two things being compared

The encoder is the same smoothed mean used everywhere else in this repo, reduced to the
one case this notebook needs: fit on the training portion, applied to held-out rows.
There is no inner nesting here because nothing is scored on training rows.

`logit` is applied before adding the two 1-way encodings. Adding probabilities would
make the additive baseline weaker than it should be, and a weak baseline is how a
diagnostic talks itself into a result.


In [ ]:
def levels(col):
    """Column as integer levels, NaN kept as its own level rather than dropped."""
    s = train[col]
    # NOT `s.dtype == object`. This pandas infers StringDtype for the text columns, so
    # that test is False for exactly the columns it is meant to catch and they fall
    # through to nanquantile. Ask whether the column is numeric instead.
    if (col in RAW_AS_LEVELS or col in CAT
            or not pd.api.types.is_numeric_dtype(s)):
        codes = pd.factorize(s, use_na_sentinel=False)[0]
    else:
        # Quantile bins from the training portion only. `duplicates="drop"` matters:
        # several columns have heavy point masses and would otherwise raise.
        edges = np.unique(np.nanquantile(s.to_numpy()[tr0], np.linspace(0, 1, N_BINS + 1)))
        codes = np.digitize(s.to_numpy(), edges[1:-1], right=False)
        codes = np.where(s.isna().to_numpy(), -1, codes)
    return np.asarray(codes, dtype=np.int64)


def encode(lv, fit_idx, apply_idx):
    """Smoothed target mean fit on fit_idx, read out on apply_idx."""
    prior = float(y[fit_idx].mean())
    df = pd.DataFrame({"v": lv[fit_idx], "y": y[fit_idx]})
    g = df.groupby("v", sort=False)["y"].agg(["sum", "count"])
    mean = (g["sum"] + prior * SMOOTH) / (g["count"] + SMOOTH)
    out = pd.Series(lv[apply_idx]).map(mean).to_numpy(dtype=np.float64)
    return np.nan_to_num(out, nan=prior)


def logit(p):
    p = np.clip(p, 1e-9, 1 - 1e-9)
    return np.log(p / (1 - p))


LV = {c: levels(c) for c in COLS}
print(f"{len(COLS)} columns, distinct levels: "
      + ", ".join(f"{c}:{len(np.unique(LV[c]))}" for c in COLS[:4]) + ", ...")

# The 1-way encodings, computed once and reused by every pair that contains them.
ONE = {c: encode(LV[c], tr0, va0) for c in COLS}
AUC1 = {c: roc_auc_score(y[va0], ONE[c]) for c in COLS}
print()
print("1-way target encoding, AUC of each column alone on fold 0:")
for c in sorted(COLS, key=lambda c: -AUC1[c]):
    print(f"  {c:<26} {AUC1[c]:.6f}")


## Stage 3. The 66 pairs, and the shuffled control beside them

Every pair gets the same treatment. `cross` is the pair of levels combined into one key,
so a cell is one combination of a bin of A and a bin of B.

The additive baseline is fit as a two-term logistic regression on the held-out rows'
own encodings, which is deliberately generous: it gives the additive model the best
coefficients it could possibly have, so any lift that survives is not an artefact of
having weighted the two columns badly.


In [ ]:
from sklearn.linear_model import LogisticRegression


def cross_key(a, b):
    """Two level vectors combined into one. Offsetting avoids collisions."""
    return (a - a.min()) * (b.max() - b.min() + 1) + (b - b.min())


def additive_auc(ea, eb):
    """Best two-term additive model in logit space, fit on the rows it scores.

    Fitting the baseline on the same rows it is scored on makes it stronger than it
    should be, not weaker, so it is the conservative choice for a lift statistic.
    Both encodings are passed in rather than looked up by name, so the control can
    hand it the shuffled column's OWN encoding instead of the real one.
    """
    X2 = np.column_stack([logit(ea), logit(eb)])
    m = LogisticRegression(max_iter=1000).fit(X2, y[va0])
    return roc_auc_score(y[va0], m.decision_function(X2))


def pair_lift(la, lb, ea=None, eb=None):
    ea = encode(la, tr0, va0) if ea is None else ea
    eb = encode(lb, tr0, va0) if eb is None else eb
    key = cross_key(la, lb)
    enc = encode(key, tr0, va0)
    ac = roc_auc_score(y[va0], enc)
    aa = additive_auc(ea, eb)
    return ac, aa, ac - aa, len(np.unique(key[tr0]))


def shuffle_within_target(lv, rng):
    """Permute a column inside each target class.

    This preserves the joint distribution of (column, target) exactly, so the column's
    own 1-way encoding is statistically unchanged and the additive baseline stays as
    strong as it was. The ONLY thing destroyed is this column's pairing with the other
    one, which is the thing under test. A plain shuffle would also destroy the column's
    own signal and would measure damage rather than the null.
    """
    out = lv.copy()
    for cls in (0, 1):
        idx = np.where(y == cls)[0]
        out[idx] = lv[rng.permutation(idx)]
    return out


rows = []
t0 = time.time()
for ca, cb in itertools.combinations(COLS, 2):
    ac, aa, lift, n_cells = pair_lift(LV[ca], LV[cb], ONE[ca], ONE[cb])
    rows.append({"a": ca, "b": cb, "auc_cross": ac, "auc_add": aa,
                 "lift": lift, "cells": n_cells})
rows.sort(key=lambda r: -r["lift"])
print(f"{len(rows)} pairs in {time.time() - t0:.0f}s")

# The negative control, on the strongest pairs so it is measured where a false
# positive would actually cost something.
rng = np.random.default_rng(SHUFFLE_SEED)
ctrl = []
for ca, cb in [(r["a"], r["b"]) for r in rows[:N_SHUFFLE]]:
    lb_shuf = shuffle_within_target(LV[cb], rng)
    ctrl.append(pair_lift(LV[ca], lb_shuf, ONE[ca], None)[2])
ctrl = np.array(ctrl)

# The control must reproduce the marginal it claims to preserve, or it is not the null
# it is advertised as. Checked rather than asserted.
_chk = encode(shuffle_within_target(LV[rows[0]["b"]], rng), tr0, va0)
print(f"control sanity: shuffled column's own AUC "
      f"{roc_auc_score(y[va0], _chk):.6f} against the real "
      f"{AUC1[rows[0]['b']]:.6f}")

print()
print(f"{'pair':<52} {'cross':>9} {'additive':>9} {'lift':>10} {'cells':>7}")
for r in rows[:15]:
    print(f"{r['a'] + ' x ' + r['b']:<52} {r['auc_cross']:>9.6f} "
          f"{r['auc_add']:>9.6f} {r['lift']:>+10.6f} {r['cells']:>7}")
print("  ...")
for r in rows[-3:]:
    print(f"{r['a'] + ' x ' + r['b']:<52} {r['auc_cross']:>9.6f} "
          f"{r['auc_add']:>9.6f} {r['lift']:>+10.6f} {r['cells']:>7}")


In [ ]:
best = rows[0]
c_mean, c_sd, c_max = ctrl.mean(), ctrl.std(ddof=1), ctrl.max()

print(f"shuffled control, {len(ctrl)} pairs: mean {c_mean:+.6f}, "
      f"sd {c_sd:.6f}, max {c_max:+.6f}")
print(f"best real pair:  {best['a']} x {best['b']}  lift {best['lift']:+.6f}")
print()

# How many real pairs clear the whole shuffled band, not just its mean. A lift that
# only beats the average of the control is not distinguishable from capacity.
above = [r for r in rows if r["lift"] > c_max]
print(f"real pairs above the control's MAXIMUM: {len(above)} of {len(rows)}")
for r in above[:10]:
    print(f"  {r['a'] + ' x ' + r['b']:<52} {r['lift']:+.6f}")

OUTSIDE = best["lift"] > c_max
print()
if not OUTSIDE:
    print("VERDICT: null. The best real pair does not clear the shuffled control, so")
    print("  the lift statistic is measuring cell count rather than interaction. The")
    print("  clause is VERIFIED and feature engineering stays rejected.")
elif best["lift"] >= LIFT_GATE:
    print(f"VERDICT: build it. Best lift {best['lift']:+.6f} clears both the control")
    print(f"  and the {LIFT_GATE:+.5f} gate. {len(above)} pairs are outside the control")
    print("  band, so 32 gets a crossed feature set built from those pairs.")
else:
    print(f"VERDICT: partial. Best lift {best['lift']:+.6f} clears the control but not")
    print(f"  the {LIFT_GATE:+.5f} gate, so this is real and small. Carry only the")
    print(f"  {min(len(above), 10)} strongest pairs into 32, not all 66.")

print()
print("This notebook writes no ledger row. It prices an idea; 32 tests it.")


In [ ]:
# The ranking is the deliverable, so it is saved rather than left in the output of a
# cell that a later kernel restart would erase.
import csv

dest = (Path("/kaggle/working") if ON_KAGGLE else LOCAL / "artifacts") / "pair_lift.csv"
with dest.open("w", newline="", encoding="utf-8") as fh:
    w = csv.DictWriter(fh, fieldnames=["a", "b", "auc_cross", "auc_add", "lift", "cells"])
    w.writeheader()
    w.writerows(rows)
print(f"wrote {dest}")
print(f"control band: [{ctrl.min():+.6f}, {ctrl.max():+.6f}]")
